<a href="https://colab.research.google.com/github/muneer-ahmad10/Natural-Language-processing/blob/main/Project_02_Mini_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
#Training Data

text = """
i love machine learning
machine learning is amazing
deep learning uses neural networks
transformers changed ai
i love transformers
ai is the future
"""

In [3]:
words=text.split()
vocab=sorted(set(words))
# vocab=sorted(set(text.split()))

# Create Word -> Number
word_to_idx={
    word : i
    for i ,word in enumerate(vocab)
}
word_to_idx



{'ai': 0,
 'amazing': 1,
 'changed': 2,
 'deep': 3,
 'future': 4,
 'i': 5,
 'is': 6,
 'learning': 7,
 'love': 8,
 'machine': 9,
 'networks': 10,
 'neural': 11,
 'the': 12,
 'transformers': 13,
 'uses': 14}

In [4]:
#Reverse the Dictionary later : number -> Word During generation
idx_to_word={
    i : word
    for  word,i in word_to_idx.items()
}
idx_to_word

{0: 'ai',
 1: 'amazing',
 2: 'changed',
 3: 'deep',
 4: 'future',
 5: 'i',
 6: 'is',
 7: 'learning',
 8: 'love',
 9: 'machine',
 10: 'networks',
 11: 'neural',
 12: 'the',
 13: 'transformers',
 14: 'uses'}

In [5]:
enumerate(vocab)

In [6]:
 word_to_idx

{'ai': 0,
 'amazing': 1,
 'changed': 2,
 'deep': 3,
 'future': 4,
 'i': 5,
 'is': 6,
 'learning': 7,
 'love': 8,
 'machine': 9,
 'networks': 10,
 'neural': 11,
 'the': 12,
 'transformers': 13,
 'uses': 14}

In [7]:
idx_to_word

{0: 'ai',
 1: 'amazing',
 2: 'changed',
 3: 'deep',
 4: 'future',
 5: 'i',
 6: 'is',
 7: 'learning',
 8: 'love',
 9: 'machine',
 10: 'networks',
 11: 'neural',
 12: 'the',
 13: 'transformers',
 14: 'uses'}

In [8]:
sequence_length = 3

inputs = []
targets = []

tokenized = [
    word_to_idx[word]
    for word in words
]

for i in range(len(tokenized)-sequence_length):

    input_seq = tokenized[i:i+sequence_length]

    target = tokenized[i+sequence_length]

    inputs.append(input_seq)

    targets.append(target)

inputs = torch.tensor(inputs)
targets = torch.tensor(targets)

In [9]:
class MiniGPT(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        num_heads,
        hidden_dim
    ):

        super().__init__()

        # embeddings
        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim
        )

        # attention
        self.attention = nn.MultiheadAttention(
            embed_dim,
            num_heads,
            batch_first=True
        )

        # feed forward
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim)
        )

        # final prediction
        self.fc = nn.Linear(
            embed_dim,
            vocab_size
        )

    def forward(self, x):

        # embeddings
        x = self.embedding(x)

        # self attention
        attn_output, _ = self.attention(
            x,
            x,
            x
        )

        # feed forward
        x = self.ff(attn_output)

        # take last token
        x = x[:, -1, :]

        # predict next word
        out = self.fc(x)

        return out

In [10]:
model = MiniGPT(
    vocab_size=len(vocab),
    embed_dim=32,
    num_heads=2,
    hidden_dim=64
)

In [11]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.01
)

In [12]:
for epoch in range(300):

    optimizer.zero_grad()

    outputs = model(inputs)

    loss = criterion(
        outputs,
        targets
    )

    loss.backward()

    optimizer.step()

    if epoch % 50 == 0:
        print(
            f"Epoch {epoch}, Loss: {loss.item()}"
        )

Epoch 0, Loss: 2.741182327270508
Epoch 50, Loss: 1.120563865697477e-06
Epoch 100, Loss: 3.5762741390499286e-07
Epoch 150, Loss: 2.622602437440946e-07
Epoch 200, Loss: 2.02655698444687e-07
Epoch 250, Loss: 1.7285341868955584e-07


In [13]:
def generate_text(start_text, length=5):

    model.eval()

    words_input = start_text.split()

    for _ in range(length):

        input_seq = [
            word_to_idx[word]
            for word in words_input[-3:]
        ]

        input_tensor = torch.tensor(
            [input_seq]
        )

        with torch.no_grad():

            output = model(input_tensor)

            predicted_idx = torch.argmax(
                output,
                dim=1
            ).item()

        predicted_word = idx_to_word[
            predicted_idx
        ]

        words_input.append(predicted_word)

    return " ".join(words_input)

In [14]:
print(
    generate_text("i love")
)

i love transformers ai is the future


In [15]:
print(
    generate_text("deep learning")
)

deep learning uses neural networks transformers changed


In [16]:
print(
    generate_text("amazing")
)

amazing deep ai love is learning
